In [1]:

import json
from pathlib import Path
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms.functional as TF
import torch

In [2]:
torch.set_num_threads(96)
torch.set_num_interop_threads(96)

In [3]:
torch.get_num_threads()

96

In [2]:
from utils.midogpp import MIDOGpp

In [3]:
from utils.midesec import MiDeSec

In [ ]:
import sqlite3
from pathlib import Path
from torch.utils.data import Dataset
from collections import defaultdict

class MiCCMCT(Dataset):
    """
    MITOS_WSI_CCMCT Dataset
    """

    def __init__(self, root_dir, sqlite_path, transforms=None):
        self.root_dir = Path(root_dir)
        self.transforms = transforms
        self.sqlite_path = sqlite_path
        self.source = "CCMCT"
        self.conn = sqlite3.connect(self.sqlite_path)
        self.cursor = self.conn.cursor()

        # -------------------------
        # 1. Load slides
        # -------------------------
        self.cursor.execute("SELECT uid, filename FROM Slides;")
        slides_data = self.cursor.fetchall()

        # slide_uid -> full image path
        self.slides = {
            uid: self.root_dir / filename
            for uid, filename in slides_data
        }

        # -------------------------
        # 2. Load class mapping
        # -------------------------
        self.cursor.execute("SELECT uid, name FROM Classes;")
        self.classes = dict(self.cursor.fetchall())

        # -------------------------
        # 3. Load annotations + labels
        # -------------------------
        self.cursor.execute("""
            SELECT A.uid, A.slide, L.class
            FROM Annotations A
            JOIN Annotations_label L
            ON A.uid = L.annoId
        """)
        ann_data = self.cursor.fetchall()
        # (anno_id, slide_uid, class_id)

        # -------------------------
        # 4. Load coordinates
        # -------------------------
        self.cursor.execute("""
            SELECT annoId, coordinateX, coordinateY
            FROM Annotations_coordinates
            ORDER BY annoId, orderIdx
        """)
        coord_data = self.cursor.fetchall()

        # Group coordinates by annotation
        coords_by_ann = defaultdict(list)
        for annoId, x, y in coord_data:
            self.coords_by_ann[annoId] = (x, y)
        
        self.samples = defaultdict(list)

        for anno_id, slide_uid, class_id in ann_data:
            if anno_id not in coords_by_ann:
                continue

            sample = {
                "label": class_id,
                "class_name": self.classes.get(class_id, None),
                "points": coords_by_ann[anno_id]
            }

            # self.samples[slide_uid].append(sample)
            if slide_uid in self.samples:
                self.samples[slide_uid]["instances"].append(sample)
            
            else:
                self.samples[slide_uid] = {
                    "image_id": slide_uid,
                    "slide_path": self.slides[slide_uid],
                    "instances" : [sample]
                }


        print(f"Loaded {len(self.ann_data)} annotations.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_data = self.samples[idx]
        return image_data


In [ ]:
import openslide
import random
import torch
import numpy as np
from torch.utils.data import Dataset
from typing import List

class PatchSampler(Dataset):
    def __init__(
        self,
        datasets: List[Dataset],
        patch_size: int = 512,
        bg_ratio: float = 0.3,
        epoch_length: int = 1000,
    ):
        self.datasets = datasets
        self.patch_size = patch_size
        self.bg_ratio = bg_ratio
        self.epoch_length = epoch_length
        self.turn = 0

    def __len__(self):
        return self.epoch_length

    def _next_dataset(self):
        self.turn = (self.turn+1)%len(self.datasets)
        return self.turn #pick the next dataset for every call

    @staticmethod
    def intersects(patch, bbox):
        px1, py1, px2, py2 = patch
        bx1, by1, bx2, by2 = bbox
        return not (px2 <= bx1 or px1 >= bx2 or py2 <= by1 or py1 >= by2)

    def _pick_patch_v1(self, dataset):
        # Sampling logic for MIDOG++ & MiDeSEC
        idx = random.randrange(len(dataset))
        sample = self.dataset[idx]

        image = sample["image"]      # [C, H, W]
        bboxes = sample["bboxes"]    # [(x1,y1,x2,y2)...]
        labels = sample["labels"]    # [1,2]
        image_id = sample["image_id"]

        _, H, W = image.shape
        P = self.patch_size

        # --- choose patch location ---
        if random.random() < self.bg_ratio:
            # background-biased random patch
            patch_x1 = random.randint(0, W - P)
            patch_y1 = random.randint(0, H - P)
        else:
            # bias around a random instance
            i = random.randrange(len(bboxes))
            x1, y1, x2, y2 = bboxes[i]
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            patch_x1 = int(cx - P // 2)
            patch_y1 = int(cy - P // 2)

            patch_x1 = max(0, min(patch_x1, W - P))
            patch_y1 = max(0, min(patch_y1, H - P))

        patch_x2 = patch_x1 + P
        patch_y2 = patch_y1 + P
        patch_box = (patch_x1, patch_y1, patch_x2, patch_y2)

        patch = image[:, patch_y1:patch_y2, patch_x1:patch_x2]

        # --- collect all instances in patch ---
        patch_bboxes = []
        patch_labels = []

        for bb, lbl in zip(bboxes, labels):
            if self.intersects(patch_box, bb):
                x1, y1, x2, y2 = bb
                patch_bboxes.append([
                    x1 - patch_x1,
                    y1 - patch_y1,
                    x2 - patch_x1,
                    y2 - patch_y1,
                ])
                patch_labels.append(lbl)

        patch = patch.float().div_(255.0)
        patch = patch[:3]

        if len(patch_bboxes) == 0:
            return (patch,  {
                            "boxes": torch.zeros((0, 4), dtype=torch.float32),
                            "labels": torch.zeros((0,), dtype=torch.long),
                            "image_id": image_id,
                            })

        return (patch,  {
                        "boxes": torch.tensor(patch_bboxes, dtype=torch.float32),
                        "labels": torch.tensor(patch_labels, dtype=torch.long),
                        "image_id": image_id,
                        })

    def _pick_patch_v2(self, dataset):
        # ---- pick slide ----
        idx = random.randrange(len(dataset))
        sample = dataset[idx]

        image_id = sample["image_id"]
        slide = openslide.OpenSlide(sample["slide_path"])
        instances = sample["instances"]  # list of { "label": int, "points": (x, y) }

        W, H = slide.level_dimensions[0]
        P = self.patch_size

        # ---- choose patch location ----
        if random.random() < self.bg_ratio or len(instances) == 0:
            # background patch
            patch_x1 = random.randint(0, W - P)
            patch_y1 = random.randint(0, H - P)
        else:
            # center on random centroid
            inst = random.choice(instances)
            cx, cy = inst["points"]

            patch_x1 = int(cx - P // 2)
            patch_y1 = int(cy - P // 2)

            patch_x1 = max(0, min(patch_x1, W - P))
            patch_y1 = max(0, min(patch_y1, H - P))

        patch_x2 = patch_x1 + P
        patch_y2 = patch_y1 + P

        # ---- read patch ----
        patch = slide.read_region(
            (patch_x1, patch_y1),
            0,
            (P, P)
        ).convert("RGB")

        patch = torch.from_numpy(np.array(patch)).permute(2, 0, 1).float().div_(255.0)

        # ---- collect centroids inside patch ----
        patch_points = []
        patch_labels = []

        for inst in instances:
            cx, cy = inst["points"]
            lbl = inst["label"]

            if patch_x1 <= cx < patch_x2 and patch_y1 <= cy < patch_y2:
                # shift to patch-local coordinates
                local_x = cx - patch_x1
                local_y = cy - patch_y1

                patch_points.append([local_x, local_y])
                patch_labels.append(lbl)

        if len(patch_points) == 0:
            return (
                patch,
                {
                    "points": torch.zeros((0, 2), dtype=torch.float32),
                    "labels": torch.zeros((0,), dtype=torch.long),
                    "image_id": image_id,
                },
            )

        return (
            patch,
            {
                "points": torch.tensor(patch_points, dtype=torch.float32),
                "labels": torch.tensor(patch_labels, dtype=torch.long),
                "image_id": image_id,
            },
        )
        return {
            image_id: image_id,
        }
        


    def __getitem__(self, _):
        turn_idx = self._next_dataset()
        dataset = self.datasets[turn_idx]
        if dataset.source == "CMC" or dataset.source == "CCMCT":
            return self._pick_patch_v2(dataset)
        else:
            return self._pick_patch_v1(dataset)
        

In [6]:
midogpp = MIDOGpp("/home/student4/PycharmProjects/DOG_MD/data/midogpp/", "/home/student4/PycharmProjects/DATA/MIDOGpp_IMAGES/MIDOGpp.json")

In [7]:
p1s = PatchSampler(midogpp)

In [ ]:
p1s[1]

(tensor([[[0.9098, 0.9176, 0.9255,  ..., 0.9255, 0.9255, 0.9255],
          [0.9176, 0.9255, 0.9373,  ..., 0.9294, 0.9255, 0.9255],
          [0.9216, 0.9294, 0.9529,  ..., 0.9333, 0.9255, 0.9255],
          ...,
          [0.6196, 0.5569, 0.4824,  ..., 0.6980, 0.6980, 0.7686],
          [0.5843, 0.5333, 0.4863,  ..., 0.6902, 0.6863, 0.7020],
          [0.5608, 0.5373, 0.5059,  ..., 0.6627, 0.6941, 0.7255]],
 
         [[0.8784, 0.8902, 0.9020,  ..., 0.8902, 0.8902, 0.8941],
          [0.9098, 0.9020, 0.8980,  ..., 0.8941, 0.8941, 0.8902],
          [0.9216, 0.9098, 0.9176,  ..., 0.8980, 0.8941, 0.8902],
          ...,
          [0.4000, 0.3490, 0.3255,  ..., 0.4784, 0.5137, 0.5137],
          [0.3843, 0.3490, 0.3216,  ..., 0.4353, 0.4667, 0.4941],
          [0.3725, 0.3608, 0.3294,  ..., 0.4431, 0.4667, 0.4941]],
 
         [[0.9255, 0.9020, 0.8824,  ..., 0.9137, 0.9098, 0.9098],
          [0.8863, 0.9059, 0.9451,  ..., 0.9137, 0.9098, 0.9098],
          [0.9176, 0.9176, 0.8980,  ...,

In [9]:
from typing import List

In [10]:
# class MemConsolidator(Dataset):
#     def __init__(self, samplers: List[PatchSampler], ratios: List[float]):
#         assert len(ratios) == len(samplers)
#         self.samplers = samplers
#         self.

In [11]:
# TODO: Make a evaluation test sampler



In [12]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2

# Load model and modify the head for custom classes
num_classes = 3

print(num_classes)

# Modify the box predictor to match the number of classes
model = fasterrcnn_resnet50_fpn_v2(weights=None,weights_backbone=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

3


In [13]:
device = "cuda:0"

In [14]:
import torch
def collate_fn(batch):
    images = []
    targets = []

    for img, target in batch:
        images.append(img)        # [C, H, W] → stackable
        target.pop('image_id')
        targets.append(target)    # dict with variable-length tensors

    images = torch.stack(images, dim=0)
    return images, targets

In [15]:
train_loader = torch.utils.data.DataLoader(
    p1s, batch_size=8, shuffle=True, collate_fn=collate_fn
)

In [16]:
import torch.optim as optim
from tqdm import tqdm
# Define optimizer
optimizer = optim.SGD(
    model.parameters(), lr=0.01, momentum=0.9, weight_decay=0.0005
)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# Training loop
model.to(device)

num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    for images, targets in tqdm(train_loader):
        targets = [
                {k: v.to(device) for k, v in t.items()}
                for t in targets
            ]
        loss_dict = model(images.to(device), targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    lr_scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {losses.item()}")

100%|██████████| 125/125 [05:32<00:00,  2.66s/it]


Epoch 1, Loss: 0.3708169758319855


100%|██████████| 125/125 [05:30<00:00,  2.65s/it]


Epoch 2, Loss: 0.17349381744861603


100%|██████████| 125/125 [05:30<00:00,  2.65s/it]


Epoch 3, Loss: 0.129086434841156


... (30 outputs left out)

100%|██████████| 125/125 [05:30<00:00,  2.65s/it]


Epoch 19, Loss: 0.11751695722341537


 62%|██████▏   | 78/125 [03:27<02:05,  2.66s/it]


KeyboardInterrupt: 

In [ ]:
for images, targets in train_loader:
    image = images[0]
    ch = image[3]
    print(ch.min().item(), ch.max().item(), ch.mean().item())
    break

In [17]:
import gc
torch.cuda.empty_cache()

In [18]:
torch.save(model, 'fasterrcnn.pt')